In [1]:
from sklearnex import patch_sklearn

patch_sklearn()
import numpy as np
import optuna
import pandas as pd
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import (
    ADASYN,
    SMOTE,
    SMOTEN,
    SMOTENC,
    SVMSMOTE,
    BorderlineSMOTE,
    KMeansSMOTE,
    RandomOverSampler,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import (
    AllKNN,
    ClusterCentroids,
    CondensedNearestNeighbour,
    EditedNearestNeighbours,
    InstanceHardnessThreshold,
    NearMiss,
    NeighbourhoodCleaningRule,
    OneSidedSelection,
    RandomUnderSampler,
    RepeatedEditedNearestNeighbours,
    TomekLinks,
)
from ipynb.fs.defs.preprocessing import get_data  # type: ignore
from optuna import create_study, create_trial
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,  # AUPRC
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from util import summary


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [2]:
# TODO K-Fold type error regularization different models
# TODO prevent overfit on train set, on val set,...

In [3]:
X_train, X_val, X_test, y_train, y_val, y_test = get_data(
    use_val=True, test_size=0.2, val_size=0.2
)

In [4]:
y_train

48204     0
12439     0
131541    0
223523    0
27369     0
         ..
111409    0
247656    0
215266    0
84324     0
94566     0
Name: Class, Length: 170884, dtype: int64

In [5]:
# oversamplers = RandomOverSampler(random_state=42, sampling_strategy=0.003)
# undersamplers = RandomUnderSampler(random_state=42, sampling_strategy=0.01)
# threshold = 0.5

In [6]:
# X_resampled, y_resampled = oversamplers.fit_resample(X_train, y_train)
# X_resampled, y_resampled = undersamplers.fit_resample(X_resampled, y_resampled)
# clf.fit(X_resampled, y_resampled)

In [7]:
# probs = clf.predict_proba(X_test)
# y_pred = (probs >= threshold).astype(int)[:, 1]
# summary(y_test, y_pred)

In [8]:
# over_samplers = {
#     "RandomUnderSampler": RandomUnderSampler,
#     "SMOTE": SMOTE,
#     "ADASYN": ADASYN,
#     "SMOTEN": SMOTEN,
#     # "SMOTENC": SMOTENC,  # type error need more work
#     "SVMSMOTE": SVMSMOTE,
#     "BorderLineSMOTE": BorderlineSMOTE,
#     "KMeansSMOTE": KMeansSMOTE,
# }

In [9]:
# under_samplers = {
#     # "ClusterCentroids": ClusterCentroids, #too slow for hyperparameters tuning
#     # "AllKNN": AllKNN, different sampling_strategy, need more work
#     # "CondensedNearestNeighbour": CondensedNearestNeighbour,# ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
#     # "EditedNearestNeighbours": EditedNearestNeighbours,# type error, need work
#     "InstanceHardnessThreshold": InstanceHardnessThreshold,
#     "NearMiss": NearMiss,
#     # "NeighbourhoodCleaningRule": NeighbourhoodCleaningRule,
#     # "OneSidedSelection": OneSidedSelection,# ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
#     "RandomUnderSampler": RandomUnderSampler,
#     # "RepeatedEditedNearestNeighbours": RepeatedEditedNearestNeighbours,
#     # "TomekLinks": TomekLinks, # ValueError: 'clean-sampling' methods do let the user specify the sampling ratio.
# }

In [10]:
def get_all_samplers(
    trial: optuna.Trial, minority_ratio: float, n_minority: int = None
):
    RANDOM_STATE = 42
    N_JOBS = -1
    SLOW_PRE_RATIO = 0.01

    def get_k(name, low=3, high=7):
        k = trial.suggest_int(name, low, high)
        if n_minority is not None:
            k = min(k, max(1, n_minority - 1))
        return k

    sampler = trial.suggest_categorical(
        "sampler",
        [
            "none",
            "RandomOver",
            "SMOTE",
            "BorderlineSMOTE",
            "ADASYN",
            "KMeansSMOTE",
            "RandomUnder",
            "NearMiss",
            "TomekLinks",
            "ENN",
            "OSS",
            "NCR",
            "SMOTEENN",
            "SMOTETomek",
        ],
    )

    if sampler == "none":
        return None

    if sampler == "RandomOver":
        ratio = trial.suggest_float("over_ratio", min(minority_ratio, 1.0), 1.0)
        return RandomOverSampler(sampling_strategy=ratio, random_state=RANDOM_STATE)

    if sampler == "SMOTE":
        ratio = trial.suggest_float("over_ratio", min(minority_ratio, 1.0), 1.0)
        k = get_k("smote_k", 3, 7)
        return SMOTE(sampling_strategy=ratio, k_neighbors=k, random_state=RANDOM_STATE)

    if sampler == "BorderlineSMOTE":
        ratio = trial.suggest_float("over_ratio", min(minority_ratio, 1.0), 1.0)
        k = get_k("border_k", 3, 7)
        return BorderlineSMOTE(
            sampling_strategy=ratio, k_neighbors=k, random_state=RANDOM_STATE
        )

    if sampler == "ADASYN":
        ratio = trial.suggest_float("over_ratio", min(minority_ratio, 1.0), 1.0)
        k = get_k("adasyn_k", 3, 7)
        return ADASYN(sampling_strategy=ratio, n_neighbors=k, random_state=RANDOM_STATE)

    if sampler == "KMeansSMOTE":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        ratio = trial.suggest_float("kmeans_over_ratio", min(pre_ratio, 1.0), 1.0)
        k = get_k("kmeans_k", 3, 7)
        thresh = trial.suggest_float("kmeans_thresh", 0.05, 0.3)
        kmeans_smote = KMeansSMOTE(
            sampling_strategy=ratio,
            k_neighbors=k,
            cluster_balance_threshold=thresh,
            random_state=RANDOM_STATE,
        )
        return ImbPipeline([("pre_rus", pre_rus), ("kmeans_smote", kmeans_smote)])

    if sampler == "RandomUnder":
        ratio = trial.suggest_float("under_ratio", min(minority_ratio, 1.0), 1.0)
        return RandomUnderSampler(sampling_strategy=ratio, random_state=RANDOM_STATE)

    if sampler == "NearMiss":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        ratio = trial.suggest_float("nearmiss_ratio", min(pre_ratio, 1.0), 1.0)
        version = trial.suggest_int("nearmiss_version", 1, 3)
        n_nei = trial.suggest_int("nearmiss_n", 3, 10)
        nm = NearMiss(
            sampling_strategy=ratio, version=version, n_neighbors=n_nei, n_jobs=N_JOBS
        )
        return ImbPipeline([("pre_rus", pre_rus), ("nm", nm)])

    if sampler == "TomekLinks":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        cleaner = TomekLinks(n_jobs=N_JOBS)
        return ImbPipeline([("pre_rus", pre_rus), ("cleaner", cleaner)])

    if sampler == "ENN":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        enn_k = trial.suggest_int("enn_k", 3, 7)
        cleaner = EditedNearestNeighbours(n_neighbors=enn_k, n_jobs=N_JOBS)
        return ImbPipeline([("pre_rus", pre_rus), ("cleaner", cleaner)])

    if sampler == "OSS":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        cleaner = OneSidedSelection(random_state=RANDOM_STATE, n_jobs=N_JOBS)
        return ImbPipeline([("pre_rus", pre_rus), ("cleaner", cleaner)])

    if sampler == "NCR":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        cleaner = NeighbourhoodCleaningRule(n_jobs=N_JOBS)
        return ImbPipeline([("pre_rus", pre_rus), ("cleaner", cleaner)])

    if sampler == "SMOTEENN":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        over_ratio = trial.suggest_float(
            "smoteenn_over_ratio", min(pre_ratio, 0.5), 0.5
        )
        smote_k = get_k("smoteenn_k", 3, 7)
        smote_step = SMOTE(
            sampling_strategy=over_ratio, k_neighbors=smote_k, random_state=RANDOM_STATE
        )
        enn_k = trial.suggest_int("smoteenn_enn_k", 3, 7)
        enn_step = EditedNearestNeighbours(n_neighbors=enn_k, n_jobs=N_JOBS)
        combine = SMOTEENN(smote=smote_step, enn=enn_step, random_state=RANDOM_STATE)
        return ImbPipeline([("pre_rus", pre_rus), ("combine", combine)])

    if sampler == "SMOTETomek":
        pre_ratio = max(SLOW_PRE_RATIO, minority_ratio)
        pre_rus = RandomUnderSampler(
            sampling_strategy=pre_ratio, random_state=RANDOM_STATE
        )
        over_ratio = trial.suggest_float(
            "smotetomek_over_ratio", min(pre_ratio, 0.5), 0.5
        )
        smote_k = get_k("smotetomek_k", 3, 7)
        smote_step = SMOTE(
            sampling_strategy=over_ratio, k_neighbors=smote_k, random_state=RANDOM_STATE
        )
        tomek_step = TomekLinks(n_jobs=N_JOBS)
        combine = SMOTETomek(
            smote=smote_step, tomek=tomek_step, random_state=RANDOM_STATE
        )
        return ImbPipeline([("pre_rus", pre_rus), ("combine", combine)])

    return None

In [11]:
def objective(trial: optuna.Trial, X, y):
    n_minority = int(np.sum(y == 1))
    n_majority = int(np.sum(y == 0))
    minority_ratio = n_minority / n_majority if n_majority > 0 else 0.1

    try:
        sampler = get_all_samplers(trial, minority_ratio, n_minority)
    except Exception as e:
        raise optuna.TrialPruned(f"Tham số sampler không hợp lệ: {e}")

    l1_ratio = trial.suggest_float("clf_l1_ratio", 0.0, 1.0)
    c_param = trial.suggest_float("clf_C", 1e-4, 1e2, log=True)
    class_weight = (
        trial.suggest_categorical("clf_class_weight", [None, "balanced"])
        if sampler is None
        else None
    )

    clf = LogisticRegression(
        l1_ratio=l1_ratio,
        C=c_param,
        solver="saga",
        class_weight=class_weight,
        max_iter=1000,
        tol=1e-3,
        random_state=42,
    )

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auprc_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        steps = [("scaler", StandardScaler())]
        if sampler is not None:
            if isinstance(sampler, ImbPipeline):
                steps.extend(sampler.steps)
            else:
                steps.append(("sampler", sampler))
        steps.append(("classifier", clf))

        pipeline = ImbPipeline(steps)

        try:
            pipeline.fit(X_train_fold, y_train_fold)
            y_pred_proba = pipeline.predict_proba(X_val_fold)[:, 1]
            auprc = average_precision_score(y_val_fold, y_pred_proba)
            auprc_scores.append(auprc)
        except Exception as e:
            raise optuna.TrialPruned(f"Lỗi khi huấn luyện Fold {fold}: {e}")

    return float(np.mean(auprc_scores))

In [12]:
study = optuna.create_study(
    storage="sqlite:///optuna.db",
    study_name="logreg_2",
    directions=["maximize"],
    load_if_exists=True,
)

[I 2026-10-19 23:45:46,441] A new study created in RDB with name: logreg_2


In [ ]:
X_arr = np.asarray(X_train)
y_arr = np.asarray(y_train)


study.optimize(lambda trial: objective(trial, X_arr, y_arr), n_trials=1000)

[I 2026-10-19 23:45:49,504] Trial 0 finished with value: 0.15460921080301515 and parameters: {'sampler': 'NearMiss', 'nearmiss_ratio': 0.5134529421424786, 'nearmiss_version': 2, 'nearmiss_n': 5, 'clf_l1_ratio': 0.48933105561070034, 'clf_C': 30.42836188386843}. Best is trial 0 with value: 0.15460921080301515.
[I 2026-10-19 23:45:54,288] Trial 1 finished with value: 0.001749724822879 and parameters: {'sampler': 'ENN', 'enn_k': 7, 'clf_l1_ratio': 0.7056757037255859, 'clf_C': 0.0002943820519139998}. Best is trial 0 with value: 0.15460921080301515.
[I 2026-10-19 23:45:56,826] Trial 2 finished with value: 0.001749724822879 and parameters: {'sampler': 'NearMiss', 'nearmiss_ratio': 0.3021958034921336, 'nearmiss_version': 2, 'nearmiss_n': 6, 'clf_l1_ratio': 0.8527143355518815, 'clf_C': 0.00029270235821460147}. Best is trial 0 with value: 0.15460921080301515.
[I 2026-10-19 23:46:00,190] Trial 3 finished with value: 0.1857192498243861 and parameters: {'sampler': 'NearMiss', 'nearmiss_ratio': 0.42

In [ ]:
test(
    sorted(
        [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE],
        key=lambda t: t.value,
        reverse=(study.direction == optuna.study.StudyDirection.MAXIMIZE),
    )[1].params
)